In [2]:
# ==========================================================
# Notebook 5: Mean Pooling and Manual Embedding Extraction
# ==========================================================

print("Notebook 5: Mean Pooling and Manual Embedding Extraction")

Notebook 5: Mean Pooling and Manual Embedding Extraction


In [3]:
import torch

from transformers import AutoTokenizer, AutoModel

In [4]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModel.from_pretrained(model_name)

print("Model Loaded!")

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Model Loaded!


In [5]:
sentence = "I love learning Artificial Intelligence."

encoded_input = tokenizer(sentence, return_tensors="pt", padding=True, truncation=True)

print(encoded_input)

{'input_ids': tensor([[ 101, 1045, 2293, 4083, 7976, 4454, 1012,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}


In [6]:
with torch.no_grad():

    model_output = model(**encoded_input)

In [7]:
print(model_output.last_hidden_state.shape)

torch.Size([1, 8, 384])


In [8]:
token_embeddings = model_output.last_hidden_state

print("Token Embedding Tensor Shape:")

print(token_embeddings.shape)

Token Embedding Tensor Shape:
torch.Size([1, 8, 384])


In [10]:
mean_embedding = torch.mean(token_embeddings, dim=1)

print(mean_embedding.shape)

mean_embedding

torch.Size([1, 384])


tensor([[ 1.1104e-01, -5.2440e-01,  3.8029e-01,  6.8140e-02,  2.4829e-01,
         -3.5276e-01,  1.4758e-01,  1.7767e-01,  3.7442e-01,  3.3292e-01,
         -1.5552e-01,  2.3874e-01, -8.6336e-02,  3.6534e-01, -4.2255e-02,
          4.1230e-02, -3.0476e-01,  3.1045e-02, -4.7414e-01, -6.3777e-01,
         -3.7708e-01,  2.2802e-01,  2.5765e-01, -2.7277e-01, -7.9434e-03,
          2.3841e-01,  6.2366e-02, -2.2840e-01, -5.4656e-02, -5.6077e-01,
         -1.7965e-01,  4.6216e-02,  2.8309e-01,  1.1840e-01, -1.6306e-01,
          2.3197e-01,  7.2189e-02, -3.0228e-01,  4.0289e-01,  9.6645e-02,
         -1.9107e-01,  3.0968e-02,  3.0865e-01, -2.1542e-01,  2.6907e-01,
          4.5728e-01, -4.4502e-01, -3.0608e-01,  5.6824e-01,  2.9024e-01,
         -3.7928e-01,  2.5854e-02, -2.6817e-01,  2.1279e-01,  9.1623e-02,
          1.1434e-01,  3.7965e-01,  1.6893e-01, -1.4435e-01, -3.6890e-01,
         -1.0338e-01, -1.4693e-01, -1.5465e-01,  2.7431e-01,  1.1745e-01,
          1.6140e-01, -1.2254e-01,  8.

In [11]:
def mean_pooling(model_output, attention_mask):

    token_embeddings = model_output.last_hidden_state

    input_mask_expanded = (
        attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    )

    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, dim=1)

    sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)

    return sum_embeddings / sum_mask

In [14]:
sentence_embedding = mean_pooling(model_output, encoded_input["attention_mask"])

print(sentence_embedding.shape)

torch.Size([1, 384])


In [15]:
toy_embeddings = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])

sentence_embedding = torch.mean(toy_embeddings, dim=0)

print(sentence_embedding)

tensor([3., 4.])


In [16]:
toy_embeddings = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])

sentence_embedding = torch.mean(toy_embeddings, dim=0)

print(sentence_embedding)

tensor([3., 4.])


In [17]:
sample = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]])

mean_pool = torch.mean(sample, dim=0)

max_pool = torch.max(sample, dim=0).values

cls_pool = sample[0]

print("CLS Pooling :", cls_pool)
print("Mean Pooling:", mean_pool)
print("Max Pooling :", max_pool)

CLS Pooling : tensor([1., 2., 3.])
Mean Pooling: tensor([4., 5., 6.])
Max Pooling : tensor([7., 8., 9.])


In [18]:
def get_sentence_embedding(sentence, tokenizer, model):

    encoded = tokenizer(sentence, return_tensors="pt", padding=True, truncation=True)

    with torch.no_grad():

        output = model(**encoded)

    embedding = mean_pooling(output, encoded["attention_mask"])

    return embedding

In [19]:
embedding = get_sentence_embedding("I love learning AI.", tokenizer, model)

print(embedding.shape)

torch.Size([1, 384])


In [20]:
sentence1 = "I love machine learning."
sentence2 = "I enjoy studying AI."

emb1 = get_sentence_embedding(sentence1, tokenizer, model)

emb2 = get_sentence_embedding(sentence2, tokenizer, model)

similarity = torch.nn.functional.cosine_similarity(emb1, emb2)

print("Cosine Similarity:", similarity.item())

Cosine Similarity: 0.5514793395996094
